In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## 1. LEFT OUTER JOIN

In [2]:
%%sql

INSERT INTO country(country)
VALUES ('New Land');

++
||
++
++

In [5]:
%%sql

SELECT c.country, ci.city
FROM country c
INNER JOIN city ci
ON c.country_id = ci.country_id

LIMIT 5;

country,city
Afghanistan,Kabul
Algeria,Batna
Algeria,Béchar
Algeria,Skikda
American Samoa,Tafuna


`New Land`は取得されない。`INNER JOIN`は、両方のテーブルに存在するデータのみを取得するためである。

| country | city |
|---|---|
| Afghanistan | あり |
| Algeria | あり |
| ... | ... |
| New Land | なし |

`LEFT OUTER JOIN`を使用すると、都市がない場合、`city`カラムは`NULL`として取得される。

In [11]:
%%sql

SELECT c.country, ci.city
FROM country c
LEFT JOIN city ci
    ON c.country_id = ci.country_id
ORDER BY ci.city IS NULL DESC

LIMIT 5;

country,city
New Land,None
Afghanistan,Kabul
Algeria,Batna
Algeria,Béchar
Algeria,Skikda


- `OUTER`は省略して、`LEFT JOIN`を使用できる。

- `LEFT JOIN`は、左側のテーブル（`country`）のすべてのデータを先に取得する。  
その後、右側のテーブル（`city`）から一致するデータを探す。

- 都市がある場合 → 一緒に出力<BR>都市がない場合 → `NULL`として出力

#### # 都市が登録されていない国を取得する

In [ ]:
%%sql

SELECT co.country, ci.country_id
FROM country co 
LEFT JOIN city ci
    ON ci.country_id = co.country_id
WHERE ci.country_id IS NULL;

country,country_id
New Land,None


#### # LEFT JOINを使用する場面

基準となるテーブルのデータを、対応するデータが存在しない行も含めてすべて取得したい場合に使用する。

- 会員一覧  
  → 注文履歴がない会員も含める

- 学生一覧  
  → 成績がない学生も含める

- 部署一覧  
  → 社員がいない部署も含める

- カテゴリ一覧  
  → 映画が登録されていないカテゴリも含める

## 2. 中間テーブルを含む3つのテーブルを結合する

#### 1) `category`テーブルに新しいカテゴリを追加する

In [15]:
%%sql

INSERT INTO category (name)
VALUES ('KContents');

++
||
++
++

#### 2) INNER JOIN 取得

In [18]:
%%sql

SELECT c.name AS category, f.title
FROM category c
INNER JOIN film_category fc
    ON c.category_id = fc.category_id
INNER JOIN film f
    ON fc.film_id = f.film_id

LIMIT 5;

category,title
Action,AMADEUS HOLY
Action,AMERICAN CIRCUS
Action,ANTITRUST TOMATOES
Action,ARK RIDGEMONT
Action,BAREFOOT MANCHURIAN


- `KContents`は取得されない。  
- `KContents`には、`film_category`テーブルに関連付けられたデータがないためである。

### データの状態

#### # `category`テーブル

| category_id | name |
|---:|---|
| 17 | KContents |

#### # `film_category`テーブル

| film_id | category_id |
|---:|---:|
| ... | 1 |
| ... | 2 |

`category_id = 17`に対応するデータが、`film_category`テーブルに存在しない。

#### 3) LEFT OUTER JOINで取得する

In [20]:
%%sql

SELECT
    c.name AS category,
    f.title
FROM category AS c
LEFT OUTER JOIN film_category AS fc
    ON c.category_id = fc.category_id
LEFT OUTER JOIN film AS f
    ON fc.film_id = f.film_id

LIMIT 5;

category,title
Action,AMADEUS HOLY
Action,AMERICAN CIRCUS
Action,ANTITRUST TOMATOES
Action,ARK RIDGEMONT
Action,BAREFOOT MANCHURIAN


`category`テーブルのすべてのカテゴリを取得する。

`film_category`テーブルに関連付けられた映画がない場合、`f.title`は`NULL`として表示される。

#### IFNULL()を使用してNULLを別の値に変更する

In [22]:
%%sql

SELECT
    c.name AS category,
    IFNULL(f.title, '登録された映画なし') AS title
FROM category AS c
LEFT OUTER JOIN film_category AS fc
    ON c.category_id = fc.category_id
LEFT OUTER JOIN film AS f
    ON fc.film_id = f.film_id

LIMIT 5;

category,title
Action,AMADEUS HOLY
Action,AMERICAN CIRCUS
Action,ANTITRUST TOMATOES
Action,ARK RIDGEMONT
Action,BAREFOOT MANCHURIAN


`f.title`が`NULL`の場合、`登録された映画なし`と表示する。

## 3. CROSS JOIN

学生テーブルと科目テーブルから、成績表を出力する場合の例。

#### 1) `student`テーブルを作成する

In [23]:
%%sql

CREATE TABLE student (
    student_id INT PRIMARY KEY,
    student_name VARCHAR(30)
);

++
||
++
++

#### 2) `subject`テーブルを作成する

In [24]:
%%sql

CREATE TABLE subject (
    subject_id INT PRIMARY KEY,
    subject_name VARCHAR(30)
);

++
||
++
++

#### 3) 学生データを追加する

In [25]:
%%sql

INSERT INTO student VALUES
(1, '千尋'),
(2, 'ハウル'),
(3, 'ソフィー');

++
||
++
++

In [26]:
%%sql

INSERT INTO subject VALUES
(101, 'SQL'),
(102, 'Python'),
(103, 'Power BI');

++
||
++
++

#### # CROSS JOIN

In [27]:
%%sql

SELECT s.student_name,
       sub.subject_name
FROM student s
CROSS JOIN subject sub;

student_name,subject_name
ソフィー,SQL
ハウル,SQL
千尋,SQL
ソフィー,Python
ハウル,Python
千尋,Python
ソフィー,Power BI
ハウル,Power BI
千尋,Power BI
